# Avionics, operations, and program

Use this notebook to try a simple sequence, find calculation inputs, and
locate subsystem activities. Hardware selections and limits are still missing
for many activities. The example below is **not vehicle verification evidence**.


In [1]:
import pandas as pd
from IPython.display import display

from hyperion.systems import state_machine
from hyperion.verification import activities, input_contracts

pd.set_option("display.max_colwidth", 100)


## 1. Try a guarded sequence

The sequence stays in HOLD until `all_ready` is true. An anomaly returns it
to HOLD. Missing readiness data must not release the hold. Edit the event
list to try another sequence; these are demonstration rules, not approved
launch rules.


In [2]:
events = [
    {"time_s": 0, "event": "ready", "signals": {"all_ready": False}},
    {"time_s": 1, "event": "ready", "signals": {"all_ready": True}},
    {"time_s": 2, "event": "anomaly", "signals": {}},
    {"time_s": 3, "event": "ready", "signals": {}},
]
transitions = [
    {"from": "HOLD", "event": "ready", "to": "READY", "guards": {"all_ready": True}},
    {"from": "READY", "event": "anomaly", "to": "HOLD", "guards": {}},
]
example = state_machine(
    initial_state="HOLD",
    events=events,
    transitions=transitions,
    forbidden_states=["UNAUTHORIZED_RELEASE"],
)
display(pd.DataFrame(example["history"]))


,time_s,previous,state,event
0,0.0,HOLD,HOLD,ready
1,1.0,HOLD,READY,ready
2,2.0,READY,HOLD,anomaly
3,3.0,HOLD,HOLD,ready


## 2. Find a calculation's inputs

Change `CALCULATION` to inspect `systems_rf_link`, `systems_power_energy`,
`systems_critical_path`, or another entry in
[calculation_inputs.csv](data/calculation_inputs.csv).

Add approved values and source references to
[verification_inputs.json](config/verification_inputs.json) for a registered
activity. [systems_examples.json](config/systems_examples.json) contains
separate examples; the verification run does not use them.


In [3]:
CALCULATION = "systems_power_energy"
calculations = input_contracts().set_index("calculation")
selected = calculations.loc[CALCULATION]
print(selected["description"])
display(pd.DataFrame({"Required input": selected["required_inputs"].split(", ")}))


Electrical energy budget; loads contain power_w, duration_s, duty_fraction; capacity Wh, transfer times s.


,Required input
0,loads
1,capacity_wh
2,usable_fraction
3,distribution_efficiency
4,reserve_fraction
5,transfer_time_s
6,allowed_transfer_time_s


## 3. Find your subsystem's work

Change the prefix below to select requirements. For example, `AVI.3.COM`
selects communications. The [Verification register](04_Verification_Register.ipynb)
shows current results and missing evidence for every activity.


In [4]:
REQUIREMENT_PREFIX = "AVI.3.COM"
register = activities()
selected_activities = register[
    register.requirement_id.str.startswith(REQUIREMENT_PREFIX)
]
columns = {
    "activity_id": "Activity ID",
    "activity": "Required work",
    "kernel": "Calculation",
    "open_parameters": "Unresolved inputs or limits",
}
display(
    selected_activities[list(columns)]
    .rename(columns=columns)
    .fillna("External evidence")
)


,Activity ID,Required work,Calculation,Unresolved inputs or limits
411,AVI.3.COM.01.A01,RF link budget and antenna-visibility sweep.,systems_rf_link,TBD-AVI-01
412,AVI.3.COM.01.A02,Radio-frequency and link-performance tests.,External evidence,TBD-AVI-01
413,AVI.3.COM.02.A01,Telemetry packet-rate and bandwidth budget.,systems_telemetry,
414,AVI.3.COM.02.A02,Telemetry continuity functional tests.,External evidence,
415,AVI.3.COM.03.A01,Engine-command telemetry packet-rate budget.,systems_telemetry,
416,AVI.3.COM.03.A02,Command-to-telemetry end-to-end tests.,External evidence,
417,AVI.3.COM.04.A01,Link-recovery state-transition and timing model.,systems_state_logic,"TBD-AVI-23, TBD-AVI-24"
418,AVI.3.COM.04.A02,Communication-outage fault-injection tests.,External evidence,"TBD-AVI-23, TBD-AVI-24"
419,AVI.3.COM.05.A01,GNSS satellite-visibility geometry sweep.,systems_satellite_visibility,TBD-AVI-27
420,AVI.3.COM.05.A02,GNSS receiver update-rate performance tests.,External evidence,TBD-AVI-27
